# Experimentação entre as bibliotecas Pandas e Polars

## Identificação

- **Aluno**: Matheus Duarte da Silva
- **Matrícula**: 211062277
- **Disciplina**: Metodologia Científica (CIC0200 - 2026.1)
- **Professor**: Ricardo Lopes de Queiroz

## Objetivo

O objetivo deste notebook é comparar o desempenho das bibliotecas Pandas e Polars em operações comuns de manipulação de dados. 
Serão realizadas operações como leitura de arquivos, filtragem, agregação e junção, e os tempos de execução serão registrados para análise.

O resultado dessa experimentação e da análise servirá como base para a elaboração de um pequeno artigo científico para a matéria de Metodologia Científica.
Por se tratar de algo de pequena escala, o artigo não seguirá um formato rígido, mas sim uma estrutura que permita apresentar os resultados de forma clara e simples.

---

# Realização dos experimentos

## Importando as bibliotecas necessárias

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
import polars as pl

## Definindo Variáveis Globais

In [ ]:
DATASET_PATH = "./raw_data/example_dataset.csv"  # Caminho para o dataset

IS_DATASET_CREATED = os.path.exists(DATASET_PATH)

pandas_benchmark_results: list[dict[str, dict[str, float]]] = []
polars_benchmark_results: list[dict[str, dict[str, float]]] = []

## Criando um dataset de exemplo

In [ ]:
if not IS_DATASET_CREATED:

    columns_to_load = ["id", "name", "date", "sensor_status", "category", "latitude", "longitude"]
    num_rows = 73_000_000

    data = {
        "id": np.arange(num_rows),
        "name": np.random.choice(["Alice", "Beto", "Carlos", "Davi"], num_rows),
        "date": np.arange(
            np.datetime64("2026-01-01"),
            np.datetime64("2026-01-01") + np.timedelta64(num_rows, "D"),
            np.timedelta64(1, "D"),
        ),
        "sensor_status": np.random.choice(["OK", "FAIL", "WARN", "ERROR", None], num_rows),
        "category": np.random.choice(["A", "B", "C"], num_rows),
        "latitude": np.random.uniform(-90, 90, num_rows),
        "longitude": np.random.uniform(-180, 180, num_rows),
    }

    df = pl.DataFrame(data)
    df.write_csv(DATASET_PATH)

In [ ]:
df = None

## Carregando o Dataset

### Carregando o dataset com Pandas

In [ ]:
# Carregando o dataset com Pandas
start_time_pandas = time.time()

try: 

    pandas_df = pd.read_csv(DATASET_PATH)

except MemoryError as mem_err:

    print(f"Erro de memória ao carregar o dataset com Pandas: {mem_err}")

except Exception as err:

    print(f"Erro ao carregar o dataset com Pandas: {err}")

end_time_pandas = time.time()

print(f"Tempo de carregamento com Pandas: {(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "load_dataset": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
# Informações sobre o DataFrame
print("Informações do DataFrame Pandas:")
print(pandas_df.info())

In [ ]:
# Calcular valores nulos
start_time_pandas = time.time()

pandas_df.isnull().sum()

end_time_pandas = time.time()

print(f"Tempo para contar valores nulos com Pandas: " \
      f"{(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "count_nulls": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
# Removendo Nulos
start_time_pandas = time.time()

pandas_df.dropna(inplace=True)

end_time_pandas = time.time()

print(f"Tempo para remover valores nulos com Pandas: " \
      f"{(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "drop_nulls": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
# Criar coluna com média de latitude e longitude
start_time_pandas = time.time()

pandas_df["lat_long_mean"] = (pandas_df["latitude"] + pandas_df["longitude"]) / 2

end_time_pandas = time.time()

print(f"Tempo para criar coluna com média de latitude e longitude com Pandas: " \
      f"{(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "create_column": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
# Filtro com condições compostas
start_time_pandas = time.time()

filtered_pandas_df = pandas_df[
    (pandas_df["sensor_status"] == "FAIL")
    & (pandas_df["latitude"] > 0)
]

end_time_pandas = time.time()

print(f"Tempo para filtrar com Pandas: "
      f"{(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "filter": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
filtered_pandas_df = None

In [ ]:
# Groupby e agregação
start_time_pandas = time.time()

grouped_pandas_df = pandas_df.groupby(
    ["category", "sensor_status"]
).agg(
    count=("id", "count"),
    max_lat=("latitude", "max"),
    min_lat=("latitude", "min")
)

end_time_pandas = time.time()

print(f"Tempo para groupby e agregação com Pandas: " \
      f"{(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "groupby_agg": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
grouped_pandas_df = None

In [ ]:
# Ordenação
start_time_pandas = time.time()

sorted_pandas_df = pandas_df.sort_values(by="date", ascending=False)

end_time_pandas = time.time()

print(f"Tempo para ordenar com Pandas: " \
      f"{(end_time_pandas - start_time_pandas):.4f} segundos")

In [ ]:
pandas_benchmark_results.append(
    {
        "sort": {
            "start": start_time_pandas,
            "end": end_time_pandas,
            "duration": end_time_pandas - start_time_pandas,
        }
    }
)

In [ ]:
sorted_pandas_df = None

In [ ]:
# Liberando a memória
pandas_df = None  

---

### Carregando o dataset com Polars

In [ ]:
# Carregando o dataset com Polars
start_time_polars = time.time()

try: 
    
    polars_df = pl.read_csv(DATASET_PATH)

except MemoryError as mem_err:
    
    print(f"Erro de memória ao carregar o dataset com Polars: {mem_err}")

except Exception as err:
    
    print(f"Erro ao carregar o dataset com Polars: {err}")

end_time_polars = time.time()

print(f"Tempo de carregamento com Polars: "\
      f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "load_dataset": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
# Informações sobre o DataFrame
print("\nInformações do DataFrame Polars:")
print(polars_df)

In [ ]:
# Contando valores nulos
start_time_polars = time.time()

polars_df.null_count()

end_time_polars = time.time()

print(f"Tempo para contar valores nulos com Polars: " \
      f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "count_nulls": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
# Removendo Nulos
start_time_polars = time.time()

polars_df = polars_df.drop_nulls()

end_time_polars = time.time()

print(f"Tempo para remover valores nulos com Polars: " \
        f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "drop_nulls": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
# Calcular média de latitude e longitude
start_time_polars = time.time()

polars_df = polars_df.with_columns(
    ((pl.col("latitude") + pl.col("longitude")) / 2).alias("lat_long_mean")
)

end_time_polars = time.time()

print(f"Tempo para criar coluna com média de latitude e longitude com Polars: " \
        f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "create_column": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
# Filtro com condições compostas
start_time_polars = time.time()

filtered_polars_df = polars_df.filter(
    (pl.col("sensor_status") == "FAIL") 
    & (pl.col("latitude") > 0)
    )

end_time_polars = time.time()

print(f"Tempo para filtrar com Polars: " \
      f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "filter": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
filtered_polars_df = None 

In [ ]:
# Groupby e agregação
start_time_polars = time.time()

grouped_polars_df = polars_df.group_by(
    ["category", "sensor_status"]
).agg([
    pl.len().alias("count"),
    pl.col("latitude").max().alias("max_latitude"),
    pl.col("latitude").min().alias("min_latitude")
])

end_time_polars = time.time()

print(f"Tempo para groupby e agregação com Polars: " \
      f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "groupby_agg": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
grouped_polars_df = None

In [ ]:
# Ordenação
start_time_polars = time.time()

sorted_polars_df = polars_df.sort("date", descending=True)

end_time_polars = time.time()

print(f"Tempo para ordenação com Polars: " \
      f"{(end_time_polars - start_time_polars):.4f} segundos")

In [ ]:
polars_benchmark_results.append(
    {
        "sort": {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": end_time_polars - start_time_polars,
        }
    }
)

In [ ]:
sorted_polars_df = None

In [ ]:
# Liberando a memória
polars_df = None

---

## Lazy Evaluation

In [ ]:
start_time_polars = time.time()

# Construindo o pipeline otimizado
lazy_query = (
    pl.scan_csv(DATASET_PATH)
    .filter(
        pl.col("sensor_status") != "OK"
    ) 
    .with_columns(
        ((pl.col("latitude") + pl.col("longitude")) / 2).alias("lat_lon_mean")
    )
    .group_by(["category"])
    .agg(
        [
            pl.len().alias("rows_count"),
            pl.col("lat_lon_mean").mean().alias("global_position_mean"),
        ]
    )
    .sort("rows_count", descending=True)
)

resultado_final_lazy = lazy_query.collect()

end_time_polars = time.time()

total_time_polars = end_time_polars - start_time_polars

print(f"Pipeline Inteiro Executado em: {total_time_polars:.4f} segundos")

print("\nResultado da Query Otimizada:")
print(resultado_final_lazy)

In [ ]:
polars_lazy_benchmark_results = [{
    "total":
        {
            "start": start_time_polars,
            "end": end_time_polars,
            "duration": total_time_polars,
        }
}]

---

## Exportando os resultados

In [ ]:
polars_json_path = "./perf_times/polars_benchmark_results.json"
with open(polars_json_path, "w") as f:

    json.dump(polars_benchmark_results, f, indent=4)


pandas_json_path = "./perf_times/pandas_benchmark_results.json"
with open(pandas_json_path, "w") as f:

    json.dump(pandas_benchmark_results, f, indent=4)


polars_lazy_benchmark_results_json_path = "./perf_times/polars_lazy_benchmark_results.json"
with open(polars_lazy_benchmark_results_json_path, "w") as f:

    json.dump(polars_lazy_benchmark_results, f, indent=4)

---